<a href="https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/shivam25th/flyrank-1st.git"
REPO_DIR = "/content/flyrank-1st"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

print("Repository cloned successfully!")
print(os.listdir(REPO_DIR))

Repository cloned successfully!
['submission', 'DATA_USE.md', 'notebooks', '.github', 'outputs', 'GUIDE.md', '01_first_look_and_discovery.ipynb', '03_working_with_the_full_release.ipynb', 'requirements.txt', 'CLAUDE.md', 'docs', '02_your_first_readable_model.ipynb', 'scripts', 'README.md', 'skills', 'data', 'SETUP.md', 'AGENTS.md', '.gitignore', 'LICENSE', 'work', '.git']


In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank-1st/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.shape[1])

Dataset shape: (30000, 44)
Columns: 44


In [ ]:
DATA_PATH = "/content/flyrank-1st/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

## 1. My lane as an ML task (type)

### Task type: Classification

I frame this as a binary classification problem because the question is whether a content page belongs to the observed declining group or not. The business output can later be turned into a ranked review queue using the model's predicted probabilities.

Classification is a useful starting point because it gives a clear yes/no target while still allowing pages to be ranked by how strongly the model believes they belong to the declining class.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the binary classification target from the starter dataset.
# Define the binary classification target from the starter dataset.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Target counts:")
print(
    df["is_declining_label"]
    .value_counts()
    .rename(index={0: "not_declining", 1: "declining"})
)

print("\nDeclining rate:",
      round(df["is_declining_label"].mean(), 3))

Target counts:
is_declining_label
declining        16262
not_declining    13738
Name: count, dtype: int64

Declining rate: 0.542


## 2. Target or proxy

### Target: `is_declining_label`

The target is a binary label created from the starter dataset's `trend_direction` field. A page is labeled 1 when `trend_direction` is `down`, and 0 otherwise.

This is a defined label in the supplied dataset rather than a future outcome that I independently observed after the prediction point. Therefore, model performance will be described as performance against this defined label, not as proof that a page will decline in the future.

I will not use `trend_direction` or `trend_pct` as model features because they define or reveal the outcome and could cause target leakage.

In [ ]:
target = "is_declining_label"

excluded_from_features = [
    "trend_direction",
    "trend_pct"
]

print("Target:", target)

print("\nTarget distribution:")
print(
    df[target]
    .value_counts()
    .rename(index={
        0: "not_declining",
        1: "declining"
    })
)

print("\nFields excluded because they define/reveal the target:")
print(excluded_from_features)

Target: is_declining_label

Target distribution:
is_declining_label
declining        16262
not_declining    13738
Name: count, dtype: int64

Fields excluded because they define/reveal the target:
['trend_direction', 'trend_pct']


## 3. Success metric

### Primary metric: Precision@50

I will use Precision@50 as the primary success metric because the business output is a prioritized review queue. Precision@50 measures the fraction of the top 50 pages selected by the method that belong to the defined declining class.

A higher Precision@50 means that a larger share of the limited review capacity is directed toward pages matching the defined declining label.

The learned model should be judged against the fixed baseline using the same data, split, and metric. I will not consider a model better simply because it is more complex.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k_labels = labels[order[:k]]

    return top_k_labels.mean()


# Simple baseline:
# stale + visible pages receive their impressions as the score.
df["baseline_score"] = (
    (
        (df["days_since_last_update"] >= 180) &
        (df["impressions_90d"] >= 500)
    ).astype(int)
    * df["impressions_90d"]
)

baseline_p50 = precision_at_k(
    df["baseline_score"],
    df["is_declining_label"],
    k=50
)

print("Baseline Precision@50:", round(baseline_p50, 3))
print("Declining base rate:", round(df["is_declining_label"].mean(), 3))

Baseline Precision@50: 0.68
Declining base rate: 0.542


## 4. The unit of analysis, as a real dataframe
### Unit of analysis: one anonymized content item/page

Each row represents one anonymized content item or page and contains its search-performance and content attributes.

The `content_id` identifies the content item, while `client_id` identifies the anonymized client associated with it. These identifiers help describe the data structure, but they should not be treated as ordinary predictive features.

In [ ]:
print("Number of rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

print("\nExample rows:")
print(
    df[
        [
            "content_id",
            "client_id",
            "impressions_90d",
            "avg_position",
            "ctr",
            "trend_direction"
        ]
    ].head()
)

Number of rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0

Example rows:
             content_id          client_id  impressions_90d  avg_position  \
0  content_304f48230142  client_f369cb89fc             3803          10.6   
1  content_a1fb4e703a9e  client_4e07408562            15320          20.3   
2  content_9aa793d4d895  client_7f2253d7e2            12581          36.5   
3  content_331d6c4de07b  client_19581e27de            11751           6.2   
4  content_d99b7a2d90ca  client_3fdba35f04            19140          44.0   

    ctr trend_direction  
0  0.76            down  
1  0.05            down  
2  0.09            down  
3  0.49          stable  
4  0.13            down  


## 5. Why ML beats a fixed rule here
### Why use ML?

A fixed rule such as "stale AND visible" is transparent and useful as a baseline, but it uses only a small number of thresholds. Page performance can depend on several signals at the same time, including impressions, CTR, average position, content age, and update recency.

A learned model can combine these signals and estimate the probability that a page belongs to the defined declining group. These probabilities can then be used to rank pages for human review.

However, greater complexity is not automatically better. I will consider ML useful only if it improves the agreed metric, Precision@50, compared with the fixed baseline under the same evaluation design.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compare the number of signals available to a simple baseline.

baseline_features = [
    "days_since_last_update",
    "impressions_90d"
]

candidate_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

print("Baseline uses:", len(baseline_features), "signals")
print("Candidate ML model can use:", len(candidate_features), "signals")

print("\nBaseline signals:")
print(baseline_features)

print("\nCandidate ML signals:")
print(candidate_features)


Baseline uses: 2 signals
Candidate ML model can use: 6 signals

Baseline signals:
['days_since_last_update', 'impressions_90d']

Candidate ML signals:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`